# Set operations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-07-set-operations.ipynb)

A set operation combines the **results** of two queries, row by row. `UNION` stacks them and removes duplicates,
`UNION ALL` stacks them and keeps duplicates, `INTERSECT` keeps rows present in both, and `EXCEPT` keeps rows
present in the first but not the second.

Set operations are the answer to questions such as "customers who ordered in January *and* February" and
"customers who ordered in February *but not* January". This chapter explains the rules, the variants, and the
NULL behaviour interviewers ask about.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

### The data

`customers` has 8 rows and `orders` has 14 rows, all dated January to March 2023. This chapter compares the
customers who ordered in January with those who ordered in February. "Ordered" means any order row, whatever its
`status`, so customer 2's cancelled February order counts as a February order.

Worked out by hand from `orders`:

| Month    | Order ids     | Customer ids   |
|----------|---------------|----------------|
| January  | 1, 4, 7, 11   | 1, 2, 3, 5     |
| February | 2, 5, 8, 9, 13 | 1, 2, 3, 4, 6 |

Customers 7 and 8 have no orders at all, so they appear in neither list.

## 1. The rules

A **set operation** takes two complete query results and combines them into one result. It does not look inside
the queries; it only sees the rows they return. Compare this with a join, which combines *columns* from two tables
side by side. A set operation combines *rows*, one result stacked on or compared with the other.

Four rules follow from that:

1. Both queries must return the **same number of columns**.
2. Columns are matched by **position**, not by name: the first column of the left query lines up with the first
   column of the right query, and so on.
3. Matched columns must have **compatible types**. Spark converts both sides to a common type, for example `INT`
   and `BIGINT` become `BIGINT`.
4. The result's **column names come from the first query**. `ORDER BY` and `LIMIT` written after the last query
   apply to the whole combined result.

To compare two months, first build one list of customer ids per month. Each list is a query on `orders`.
`DISTINCT` makes each list contain a customer at most once.

### Where you'll use it

- **Combining sources:** a unified events table built from three source systems with the same column layout.
- **Comparing two versions of a table:** a migration check that the old and new pipeline return the same rows.
- **Building a list of ids from several places:** every customer id seen in orders, refunds or support tickets.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW jan_buyers AS
SELECT DISTINCT customer_id FROM orders WHERE month(order_date) = 1
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW feb_buyers AS
SELECT DISTINCT customer_id FROM orders WHERE month(order_date) = 2
""")

jan = "SELECT customer_id FROM jan_buyers ORDER BY customer_id"
feb = "SELECT customer_id FROM feb_buyers ORDER BY customer_id"
spark.sql(jan).show()
spark.sql(feb).show()
assert spark.sql(jan).collect() == [(1,), (2,), (3,), (5,)]
assert spark.sql(feb).collect() == [(1,), (2,), (3,), (4,), (6,)]

### Column names come from the first query

The left query names its column `buyer`, the right query names it `customer_id`. The result keeps `buyer`. The
`ORDER BY` at the end sorts the combined rows, so it must use the name from the first query.

In [ ]:
q = """
SELECT customer_id AS buyer FROM jan_buyers
UNION ALL
SELECT customer_id FROM feb_buyers
ORDER BY buyer
"""
result = spark.sql(q)
result.show()
assert result.columns == ["buyer"]
assert [r.buyer for r in result.collect()] == [1, 1, 2, 2, 3, 3, 4, 5, 6]

### Columns are matched by position

Because matching is by position, swapping two columns in one query is not an error when the types are compatible.
The values land under the wrong heading and nothing warns you.

Here the left query returns `(order_id, amount)` and the right query returns `(amount, order_id)`. Both columns are
integers, so Spark accepts the query. Order 1 (amount 100) is correct, but the second row reports an order with id
`200` and amount `2`.

In [ ]:
q = """
SELECT order_id, amount FROM orders WHERE order_id = 1
UNION ALL
SELECT amount, order_id FROM orders WHERE order_id = 2
ORDER BY order_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 100), (200, 2)]

The safe habit is to list the same columns in the same order in every query, and never use `SELECT *` in a set
operation. The DataFrame API has `unionByName`, which matches columns by name instead. Spark SQL 4.0 has no
`UNION BY NAME` syntax.

### Column count and type errors

A different number of columns is caught before the query runs: Spark raises an `AnalysisException`.

In [ ]:
from pyspark.errors import AnalysisException

try:
    spark.sql("SELECT customer_id, name FROM customers UNION SELECT customer_id FROM orders")
    raise AssertionError("expected the query to be rejected")
except AnalysisException as error:
    assert "NUM_COLUMNS_MISMATCH" in str(error)
    print("Rejected: NUM_COLUMNS_MISMATCH")

Type conversion happens per value, while the query runs. With ANSI mode on, an `INT` column next to a `STRING`
column is converted to `BIGINT`. The string `'2'` converts to the number 2. The string `'a'` cannot convert, and
the whole query fails with `CAST_INVALID_INPUT`.

In [ ]:
converted = spark.sql("SELECT 1 AS x UNION ALL SELECT '2'")
assert converted.schema["x"].dataType.simpleString() == "bigint"
assert converted.collect() == [(1,), (2,)]

try:
    spark.sql("SELECT 1 AS x UNION ALL SELECT 'a'").collect()
    raise AssertionError("expected the query to fail")
except Exception as error:
    assert "CAST_INVALID_INPUT" in str(error)
    print("Failed at run time: CAST_INVALID_INPUT")

## 2. UNION and UNION ALL

Both stack the rows of the second query under the rows of the first. They differ in one step:

- `UNION ALL` keeps every row. The row count is the sum of the two inputs.
- `UNION` then removes duplicate rows from the combined result. Two rows are duplicates when every column is
  equal.

Worked by hand: January has 4 customers and February has 5. `UNION ALL` returns 4 + 5 = 9 rows. Customers 1, 2 and 3
appear in both lists, so `UNION` drops those 3 repeats and returns 9 - 3 = 6 rows: customers 1 to 6.

### Where you'll use it

- **Stacking partitions:** `UNION ALL` combines daily tables into one history view. The days cannot overlap, so
  removing duplicates would be wasted work.
- **Building a distinct list:** `UNION` gives every customer seen in either of two sources, each once.

In [ ]:
union_q = """
SELECT customer_id FROM jan_buyers
UNION
SELECT customer_id FROM feb_buyers
ORDER BY customer_id
"""
spark.sql(union_q).show()
assert spark.sql(union_q).collect() == [(1,), (2,), (3,), (4,), (5,), (6,)]

union_all_q = """
SELECT customer_id FROM jan_buyers
UNION ALL
SELECT customer_id FROM feb_buyers
"""
assert spark.sql(union_all_q).count() == 9

### UNION removes duplicates inside each input too

`UNION` does not only remove rows that appear on both sides. It removes every duplicate in the combined result,
including repeats inside one input. The left input below is `1, 1` and the right is `2`. `UNION` returns `1, 2`.
In effect, `A UNION B` is `SELECT DISTINCT * FROM (A UNION ALL B)`.

In [ ]:
q = """
SELECT k FROM VALUES (1), (1) AS a(k)
UNION
SELECT k FROM VALUES (2) AS b(k)
ORDER BY k
"""
assert spark.sql(q).collect() == [(1,), (2,)]

same_q = """
SELECT DISTINCT k FROM (
  SELECT k FROM VALUES (1), (1) AS a(k)
  UNION ALL
  SELECT k FROM VALUES (2) AS b(k)
)
ORDER BY k
"""
assert spark.sql(same_q).collect() == spark.sql(q).collect()

### Why UNION ALL is cheaper

To remove duplicates, the engine must bring equal rows together and keep one of each. In Spark this is an
aggregation over all columns (`HashAggregate` in the plan), and it needs a shuffle (`Exchange` in the plan): rows are
moved between workers so that equal rows meet on the same worker. `UNION ALL` only appends one result to the other,
so its plan has neither step.

`EXPLAIN` returns the physical plan as text. The two queries below read raw order rows, without `DISTINCT`, so the
only difference between their plans is the duplicate removal.

In [ ]:
def physical_plan(query):
    return spark.sql("EXPLAIN " + query).collect()[0][0]

stack = """
SELECT customer_id FROM orders WHERE month(order_date) = 1
{op}
SELECT customer_id FROM orders WHERE month(order_date) = 2
"""
union_plan = physical_plan(stack.format(op="UNION"))
union_all_plan = physical_plan(stack.format(op="UNION ALL"))
print(union_all_plan)
assert "HashAggregate" in union_plan and "Exchange" in union_plan
assert "HashAggregate" not in union_all_plan and "Exchange" not in union_all_plan

## 3. INTERSECT and EXCEPT

`INTERSECT` returns the rows that appear in both queries. `EXCEPT` returns the rows of the first query that do not
appear in the second. Like `UNION`, both compare whole rows and both remove duplicates from their result. Spark
also accepts `MINUS` as another name for `EXCEPT`.

Worked by hand with January `{1, 2, 3, 5}` and February `{1, 2, 3, 4, 6}`:

| Expression        | Result      | Meaning                               |
|-------------------|-------------|---------------------------------------|
| Jan INTERSECT Feb | 1, 2, 3     | ordered in both months                |
| Feb EXCEPT Jan    | 4, 6        | ordered in February but not January   |
| Jan EXCEPT Feb    | 5           | ordered in January but not February   |

The order matters for `EXCEPT`: `A EXCEPT B` and `B EXCEPT A` are different questions. `INTERSECT` and `UNION` give
the same rows in either order.

### Where you'll use it

- **Change detection:** rows in today's extract that are not in yesterday's (`EXCEPT`) are the new or changed records.
- **Overlap analysis:** customers who bought both products (`INTERSECT`).
- **Reconciliation tests:** `old EXCEPT new` and `new EXCEPT old` both empty means two tables hold the same distinct rows.

In [ ]:
both_q = """
SELECT customer_id FROM jan_buyers
INTERSECT
SELECT customer_id FROM feb_buyers
ORDER BY customer_id
"""
spark.sql(both_q).show()
assert spark.sql(both_q).collect() == [(1,), (2,), (3,)]

feb_not_jan_q = """
SELECT customer_id FROM feb_buyers
EXCEPT
SELECT customer_id FROM jan_buyers
ORDER BY customer_id
"""
spark.sql(feb_not_jan_q).show()
assert spark.sql(feb_not_jan_q).collect() == [(4,), (6,)]

jan_not_feb_q = """
SELECT customer_id FROM jan_buyers
MINUS
SELECT customer_id FROM feb_buyers
"""
assert spark.sql(jan_not_feb_q).collect() == [(5,)]

### EXCEPT and the anti join

A `LEFT ANTI JOIN` keeps the rows of the left table that have no match in the right table. On these lists it gives
the same answer as `EXCEPT`. The difference is what gets compared: `EXCEPT` compares every selected column, while
the anti join compares only the columns in its `ON` clause and can return any other column of the left table. That
makes the anti join easier to extend, for example to add the customer's name.

In [ ]:
q = """
SELECT f.customer_id, c.name
FROM feb_buyers f
LEFT ANTI JOIN jan_buyers j ON j.customer_id = f.customer_id
JOIN customers c ON c.customer_id = f.customer_id
ORDER BY f.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(4, "Delta Co"), (6, "Foxtrot GmbH")]
assert [(r.customer_id,) for r in spark.sql(q).collect()] == spark.sql(feb_not_jan_q).collect()

### Precedence: INTERSECT binds first

When one query mixes set operators, `INTERSECT` is evaluated before `UNION` and `EXCEPT`, in the same way that `*`
is evaluated before `+`. `UNION` and `EXCEPT` have equal precedence and run left to right.

Take `A = {1}`, `B = {2}`, `C = {3}` and the expression `A UNION B INTERSECT C`:

- With `INTERSECT` first: `A UNION (B INTERSECT C)` = `{1} UNION {}` = `{1}`.
- Read left to right it would be `(A UNION B) INTERSECT C` = `{1, 2} INTERSECT {3}` = `{}`.

Spark follows the first reading. Add parentheses whenever you mix operators, so the reader does not need to know
the rule.

In [ ]:
unbracketed = "SELECT 1 AS k UNION SELECT 2 INTERSECT SELECT 3"
bracketed = "(SELECT 1 AS k UNION SELECT 2) INTERSECT SELECT 3"
assert spark.sql(unbracketed).collect() == [(1,)]
assert spark.sql(bracketed).collect() == []

## 4. The ALL variants

Plain `INTERSECT` and `EXCEPT` treat each input as a **set**: a value is either present or absent. `INTERSECT ALL`
and `EXCEPT ALL` treat each input as a **multiset** (also called a bag): a collection in which a value can appear
several times, and the number of copies matters.

For a value that appears `m` times on the left and `n` times on the right:

| Operator        | Copies in the result |
|-----------------|----------------------|
| `UNION ALL`     | m + n                |
| `INTERSECT ALL` | the smaller of m and n |
| `EXCEPT ALL`    | m - n, or 0 if n is larger |

Worked by hand with left `1, 1, 2` and right `1`: value 1 has m = 2, n = 1, so `EXCEPT ALL` keeps 2 - 1 = 1 copy.
Value 2 has m = 1, n = 0, so it keeps 1 copy. The result is `1, 2`. Plain `EXCEPT` removes value 1 entirely,
because it is present on the right at all, and returns only `2`.

### Where you'll use it

- **Accurate counts:** `EXCEPT ALL` keeps duplicates, which matters when counting transactions rather than distinct values.
- **Row-level reconciliation:** `source EXCEPT ALL target` finds rows that were loaded fewer times than they occur in
  the source, which plain `EXCEPT` cannot see.

In [ ]:
except_all_q = """
SELECT k FROM VALUES (1), (1), (2) AS a(k)
EXCEPT ALL
SELECT k FROM VALUES (1) AS b(k)
ORDER BY k
"""
spark.sql(except_all_q).show()
assert spark.sql(except_all_q).collect() == [(1,), (2,)]

except_q = except_all_q.replace("EXCEPT ALL", "EXCEPT")
assert spark.sql(except_q).collect() == [(2,)]

`INTERSECT ALL` keeps the smaller count. Left `1, 1, 1, 2` and right `1, 1, 3`: value 1 has m = 3 and n = 2, so 2
copies remain. Values 2 and 3 appear on only one side, so 0 copies remain. The result is `1, 1`.

In [ ]:
q = """
SELECT k FROM VALUES (1), (1), (1), (2) AS a(k)
INTERSECT ALL
SELECT k FROM VALUES (1), (1), (3) AS b(k)
"""
assert spark.sql(q).collect() == [(1,), (1,)]

## 5. NULL in set operations

`NULL` means "unknown". With the `=` operator, `NULL = NULL` is not true; it is `NULL`, because two unknown values
cannot be shown to be equal. A `WHERE` clause or a join `ON` clause keeps a row only when its condition is true, so
a join on a NULL key matches nothing.

Set operations use a different comparison. They ask "are these rows **distinct**?", and for that question two NULLs
count as the same value. This comparison is called **NULL-safe**. Spark writes the NULL-safe equality operator as
`<=>`: `NULL <=> NULL` is true, and `NULL <=> 1` is false.

So two NULL rows collapse into one under `UNION`, while a join on the same two NULLs returns no rows.

### Where you'll use it

- **Comparing extracts:** NULLs compare equal in set operations, which is what a diff tool needs and differs from `=` in a `WHERE`.
- **Nullable keys:** when a join key can be NULL and NULL should match NULL, use `<=>` in the `ON` clause.

In [ ]:
union_null_q = """
SELECT x FROM VALUES (CAST(NULL AS INT)) AS a(x)
UNION
SELECT x FROM VALUES (CAST(NULL AS INT)) AS b(x)
"""
spark.sql(union_null_q).show()
assert spark.sql(union_null_q).collect() == [(None,)]

join_q = """
SELECT COUNT(*) AS matched_rows
FROM VALUES (CAST(NULL AS INT)) AS a(x)
JOIN VALUES (CAST(NULL AS INT)) AS b(y) ON a.x = b.y
"""
assert spark.sql(join_q).collect() == [(0,)]

null_safe_join_q = join_q.replace("a.x = b.y", "a.x <=> b.y")
assert spark.sql(null_safe_join_q).collect() == [(1,)]

assert spark.sql("SELECT NULL = NULL, NULL <=> NULL, NULL <=> 1").collect() == [(None, True, False)]

### EXCEPT and an anti join disagree on NULL

This difference changes real answers. Left is `1, NULL` and right is `NULL`.

- `EXCEPT` treats the two NULLs as the same row and removes it. Result: `1`.
- `LEFT ANTI JOIN ... ON a.x = b.x` finds no match for the left NULL, because `NULL = NULL` is not true. It keeps
  the NULL row. Result: `1, NULL`.

When you replace one with the other, check whether the compared columns can be NULL.

In [ ]:
except_null_q = """
SELECT x FROM VALUES (1), (CAST(NULL AS INT)) AS a(x)
EXCEPT
SELECT x FROM VALUES (CAST(NULL AS INT)) AS b(x)
"""
assert spark.sql(except_null_q).collect() == [(1,)]

anti_null_q = """
SELECT a.x
FROM VALUES (1), (CAST(NULL AS INT)) AS a(x)
LEFT ANTI JOIN VALUES (CAST(NULL AS INT)) AS b(x) ON a.x = b.x
ORDER BY a.x NULLS LAST
"""
assert spark.sql(anti_null_q).collect() == [(1,), (None,)]

## Interview notes

- **"Customers who did A but not B."** Write it with `EXCEPT`, or with a `LEFT ANTI JOIN` / `NOT EXISTS`. They give
  the same answer when the compared columns are never NULL. `EXCEPT` compares every selected column; the anti join
  compares only the `ON` columns and can return extra columns, which makes it easier to extend.
- **UNION vs UNION ALL.** The usual answer is cost: `UNION` must remove duplicates, which means a sort or a hash
  aggregation (and in Spark a shuffle) over the whole result. Use `UNION ALL` by default, and use `UNION` only when
  duplicates can occur and must be removed.
- **Matching is by position.** Column names come from the first query, and swapped columns of compatible types are
  not an error. List the same columns in the same order on every side.
- **Column count mismatch** is caught at analysis time. Spark converts the two sides to a common type, so `INT` and
  `BIGINT` combine without trouble. A value that cannot convert, such as the string `'a'` next to an integer, fails
  the whole query in ANSI mode.
- **Precedence.** `INTERSECT` is evaluated before `UNION` and `EXCEPT`. Use parentheses when mixing them.
- **ALL variants** count copies: `INTERSECT ALL` keeps the smaller count, `EXCEPT ALL` keeps the difference.
- **NULLs** are treated as equal by set operations and as not equal by `=`.

## Practice questions

1. Which customers ordered in both January and February? Sort by `customer_id`.
2. Which customers ordered in February but not in January? Sort by `customer_id`.
3. How many distinct customers ordered in January or February?
4. Which customers ordered in exactly one of the two months, January or February? Sort by `customer_id`.
5. Using `EXCEPT ALL` on the `customer_id` column of `orders`, list one row for every order that is not
   `completed`, showing only `customer_id`. Sort by `customer_id`. Why would plain `EXCEPT` return no rows?

## Answers

### Question 1

"Both months" is the overlap of the two lists, so use `INTERSECT`.

In [ ]:
q = "SELECT customer_id FROM jan_buyers INTERSECT SELECT customer_id FROM feb_buyers ORDER BY customer_id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(1,), (2,), (3,)]

### Question 2

"February but not January" keeps the February list and removes January, so February goes on the left of `EXCEPT`.

In [ ]:
q = "SELECT customer_id FROM feb_buyers EXCEPT SELECT customer_id FROM jan_buyers ORDER BY customer_id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(4,), (6,)]

### Question 3

`UNION` removes the customers who appear in both lists, so counting its rows counts each customer once.
`UNION ALL` would count customers 1, 2 and 3 twice and give 9.

In [ ]:
q = """
SELECT COUNT(*) AS distinct_buyers
FROM (SELECT customer_id FROM jan_buyers UNION SELECT customer_id FROM feb_buyers)
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(6,)]

### Question 4

"Exactly one month" is everyone who ordered in either month, minus those who ordered in both:
`{1, 2, 3, 4, 5, 6}` minus `{1, 2, 3}` = `{4, 5, 6}`. The parentheses make the order of the two steps explicit.

In [ ]:
q = """
(SELECT customer_id FROM jan_buyers UNION SELECT customer_id FROM feb_buyers)
EXCEPT
(SELECT customer_id FROM jan_buyers INTERSECT SELECT customer_id FROM feb_buyers)
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(4,), (5,), (6,)]

### Question 5

Subtract the multiset of completed orders' customer ids from the multiset of all orders' customer ids. Each
customer loses one copy per completed order, and the copies left over are the orders that are not completed:
customer 2 (cancelled), customer 4 (pending) and customer 6 (cancelled). Plain `EXCEPT` returns nothing, because
each of those customers also has at least one completed order, so their id is present on the right side.

In [ ]:
q = """
SELECT customer_id FROM orders
EXCEPT ALL
SELECT customer_id FROM orders WHERE status = 'completed'
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,), (4,), (6,)]

plain_q = q.replace("EXCEPT ALL", "EXCEPT")
assert spark.sql(plain_q).collect() == []